# Phase 3 Training (VS Code + Colab/Remote Jupyter)

Use this notebook after connecting VS Code Jupyter to your Colab/remote server:
1. **Select Kernel** in VS Code.
2. Choose **Existing Jupyter Server**.
3. Paste your server URL/token.
4. Run cells top-to-bottom.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np

In [5]:
# Detect environment (local vs Colab)
import os
try:
    from google.colab import drive
    IS_COLAB = True
    print('✓ Running on Colab')
except ImportError:
    IS_COLAB = False
    print('✓ Running locally')

✓ Running on Colab


In [9]:
# Update these as needed for your local/remote environment
# For local: use relative paths from PROJECT_ROOT
# For Colab: update DATA_ROOT to point to your uploaded/mounted data folder

# ===== MANUAL SETTING FOR REMOTE KERNEL =====
# If running in a remote Jupyter kernel, set PROJECT_ROOT manually here:
PROJECT_ROOT = Path('c:/Users/rockb/OneDrive/Desktop/Projects/BTP/Phase 2')
# Or use relative path if running locally:
# PROJECT_ROOT = Path.cwd().resolve()

# Auto-detection fallback (if PROJECT_ROOT above doesn't work)
if not (PROJECT_ROOT / 'run_phase3_colab.py').exists():
    detected_root = Path.cwd().resolve()
    if (detected_root / 'notebooks').is_dir() and (detected_root.parent / 'run_phase3_colab.py').exists():
        PROJECT_ROOT = detected_root.parent
    elif (detected_root.parent / 'run_phase3_colab.py').exists():
        PROJECT_ROOT = detected_root.parent

if not (PROJECT_ROOT / 'run_phase3_colab.py').exists():
    raise FileNotFoundError(f'run_phase3_colab.py not found at {PROJECT_ROOT}. Update PROJECT_ROOT manually above.')

# Configuration
SCENARIO = 'urban'                        # or 'rural' or None
TARGET_ACCURACY = 0.80
ANTENNA_COUNT = 64
ZF_REQUIRED_DB = 15.0
MAX_TRACKING_OFFSET_RAD = 0.12
RUN_DRL = False                           # Keep False until ZF is validated

# Data paths: adjust based on where your X.npy/y.npy or split arrays are located
if IS_COLAB:
    DATA_ROOT = Path('/content/data/phase3')  # Update if using Colab
else:
    # Local: point to where you have your data files (e.g., results/baseline or a separate data folder)
    DATA_ROOT = PROJECT_ROOT / 'results' / 'baseline' / 'data'
    # If data is elsewhere, update this path:
    # DATA_ROOT = Path('C:/path/to/your/data')

CHECKPOINT_DIR = PROJECT_ROOT / 'results' / 'phase3' / 'checkpoints'
SUMMARY_JSON = PROJECT_ROOT / 'results' / 'phase3' / 'phase3_summary.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('Data exists:', DATA_ROOT.exists())
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('SUMMARY_JSON:', SUMMARY_JSON)

FileNotFoundError: run_phase3_colab.py not found at c:/Users/rockb/OneDrive/Desktop/Projects/BTP/Phase 2. Update PROJECT_ROOT manually above.

In [ ]:
# Install runtime dependencies in the connected server kernel
requirements_path = PROJECT_ROOT / 'requirements_phase3_colab.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
print('Installed:', requirements_path)

In [ ]:
# Quick dataset sanity check: expects (N, 10, 3)
x_path = DATA_ROOT / 'X.npy'
y_path = DATA_ROOT / 'y.npy'

if x_path.exists() and y_path.exists():
    X = np.load(x_path)
    y = np.load(y_path)
    print('Found unsplit dataset')
    print('X shape:', X.shape)
    print('y shape:', y.shape)
else:
    print('X.npy/y.npy not found. Assuming pre-split arrays are present.')
    for name in ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'X_test.npy', 'y_test.npy']:
        p = DATA_ROOT / name
        print(name, '->', p.exists())

In [ ]:
# Run Phase 3 pipeline: LSTM first -> Transformer fallback if needed -> proactive ZF -> optional PPO
cmd = [
    sys.executable,
    'run_phase3_colab.py',
    '--data-root', str(DATA_ROOT),
    '--target-accuracy', str(TARGET_ACCURACY),
    '--antenna-count', str(ANTENNA_COUNT),
    '--zf-required-db', str(ZF_REQUIRED_DB),
    '--max-tracking-offset-rad', str(MAX_TRACKING_OFFSET_RAD),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--summary-json', str(SUMMARY_JSON),
]

if SCENARIO is not None:
    cmd.extend(['--scenario', str(SCENARIO)])

if RUN_DRL:
    cmd.append('--run-drl')

print('Running command:\n', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

In [ ]:
# Read and inspect summary metrics
if not SUMMARY_JSON.exists():
    raise FileNotFoundError(f'Summary not found: {SUMMARY_JSON}')

payload = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
lstm = payload['prediction']['lstm']
mit = payload['mitigation']

print('LSTM test accuracy (within 10% error):', lstm['test_within_10pct_accuracy'])
print('LSTM meets target:', lstm['meets_target'])
print('Transformer fallback executed:', payload['prediction']['transformer_executed'])
if payload['prediction']['transformer'] is not None:
    print('Transformer test accuracy:', payload['prediction']['transformer']['test_within_10pct_accuracy'])
    print('Transformer meets target:', payload['prediction']['transformer']['meets_target'])
print('Mitigation predictor used:', mit['predictor_used'])
print('ZF suppression (dB):', mit['suppression_db'])
print('ZF gate passed:', mit['zf_gate_passed'])
print('PPO trained:', mit['ppo_trained'])